# Week 10 · Day 3 — Object Detection: from Localization to YOLO

Today we go beyond *"what is in this image?"* to *"what is in it, AND where?"* We follow the classic path:

1. **Classification with localization** — one object: predict its **class + bounding box**. We build & train this ourselves.
2. **IoU** — how we measure if a predicted box is "correct." We implement it.
3. **Non-max suppression** — how detectors clean up duplicate boxes. We implement it.
4. **YOLO** — the real thing: many objects at once, in real time. We run a pretrained model.

> The first three parts are **built from scratch** so the ideas are concrete. The last part shows the industrial tool that puts it all together.

> **Kaggle GPU:** Settings → Accelerator → GPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
np.random.seed(42)
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

---
# Part 1 · Classification with Localization

**The task (one object per image):** the network outputs both a **class** *and* a **bounding box** `[bx, by, bh, bw]`.

To train this we need images *with known boxes*. We generate them: each image has **one shape** (circle, square, or triangle) at a random position and size — so we get the ground-truth box for free. This is the cleanest way to see localization work.

In [ ]:
IMG = 64
SHAPES = ["circle", "square", "triangle"]

def make_image():
    """One random shape on a blank canvas. Returns image, class idx, and box [x,y,w,h] in 0-1."""
    img = np.ones((IMG, IMG, 3), dtype=np.float32) * 0.1
    cls = np.random.randint(0, 3)
    size = np.random.randint(14, 26)                 # object size in pixels
    x = np.random.randint(2, IMG - size - 2)         # top-left corner
    y = np.random.randint(2, IMG - size - 2)
    color = np.random.rand(3) * 0.6 + 0.4
    yy, xx = np.mgrid[0:IMG, 0:IMG]
    if cls == 0:      # circle
        cx, cy, r = x + size/2, y + size/2, size/2
        mask = (xx - cx)**2 + (yy - cy)**2 <= r**2
    elif cls == 1:    # square
        mask = (xx >= x) & (xx < x+size) & (yy >= y) & (yy < y+size)
    else:             # triangle
        mask = (yy >= y) & (yy < y+size) & (np.abs(xx - (x+size/2)) <= (yy - y)/2)
    img[mask] = color
    # box as [bx, by, bw, bh] normalized 0-1 (top-left + size)
    box = np.array([x/IMG, y/IMG, size/IMG, size/IMG], dtype=np.float32)
    return img, cls, box

# build a dataset
def make_dataset(n):
    X = np.zeros((n, IMG, IMG, 3), np.float32)
    yc = np.zeros(n, np.int32)
    yb = np.zeros((n, 4), np.float32)
    for i in range(n):
        X[i], yc[i], yb[i] = make_image()
    return X, yc, yb

X_train, yc_train, yb_train = make_dataset(4000)
X_test,  yc_test,  yb_test  = make_dataset(800)
print("train:", X_train.shape, " boxes:", yb_train.shape)

In [ ]:
# look at the data with ground-truth boxes drawn on
def draw(ax, img, box, label, color="lime"):
    ax.imshow(img)
    x, y, w, h = box[0]*IMG, box[1]*IMG, box[2]*IMG, box[3]*IMG
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
    ax.set_title(label, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax in axes:
    i = np.random.randint(len(X_train))
    draw(ax, X_train[i], yb_train[i], SHAPES[yc_train[i]])
plt.suptitle("Training data: one shape each, with its ground-truth box")
plt.tight_layout(); plt.show()

## The model: one CNN, two heads

Exactly the structure from the lecture. A shared CNN feature-extractor, then **two output heads**:
- **class head** — softmax over the 3 shapes (which object).
- **box head** — 4 numbers `[bx, by, bw, bh]` with sigmoid (0–1 coordinates).

And a **two-part loss**: cross-entropy for the class + mean-squared-error for the box, added together — the network learns *what* and *where* at once.

In [ ]:
inp = keras.Input((IMG, IMG, 3))
x = keras.layers.Conv2D(16, 3, activation="relu", padding="same")(inp)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dense(128, activation="relu")(x)

class_head = keras.layers.Dense(3, activation="softmax", name="class")(x)   # what
box_head   = keras.layers.Dense(4, activation="sigmoid", name="box")(x)     # where

model = keras.Model(inp, [class_head, box_head])
model.compile(
    optimizer="adam",
    loss={"class": "sparse_categorical_crossentropy", "box": "mse"},
    loss_weights={"class": 1.0, "box": 5.0},     # weight the box loss up a bit
    metrics={"class": "accuracy"})
model.summary()

In [ ]:
history = model.fit(
    X_train, {"class": yc_train, "box": yb_train},
    validation_data=(X_test, {"class": yc_test, "box": yb_test}),
    epochs=15, batch_size=64, verbose=1)

## See it localize
Green = ground truth, red = the network's prediction. A good model draws its red box right on top of the object and names the shape.

In [ ]:
pc, pb = model.predict(X_test[:10], verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, i in zip(axes.ravel(), range(10)):
    ax.imshow(X_test[i])
    # ground truth (green)
    gx, gy, gw, gh = yb_test[i]*IMG
    ax.add_patch(patches.Rectangle((gx, gy), gw, gh, fill=False, edgecolor="lime", linewidth=2))
    # prediction (red)
    px, py, pw, ph = pb[i]*IMG
    ax.add_patch(patches.Rectangle((px, py), pw, ph, fill=False, edgecolor="red", linewidth=2))
    ax.set_title(f"pred: {SHAPES[pc[i].argmax()]}", fontsize=9); ax.axis("off")
plt.suptitle("green = truth   |   red = prediction")
plt.tight_layout(); plt.show()

print("class accuracy:", (pc.argmax(1) == yc_test[:10]).mean())

**That's localization.** The same CNN you know, with a second head that regresses 4 numbers, trained with a combined loss. Everything else in detection builds on this idea.

---
# Part 2 · Intersection over Union (IoU)

How do we score a predicted box? **IoU** = area of overlap ÷ area of union of the predicted and true boxes. It's 1.0 for a perfect match, 0 for no overlap. A detection is usually called "correct" if **IoU ≥ 0.5**.

In [ ]:
def iou(box1, box2):
    """Boxes as [x1, y1, x2, y2] (corners). Returns intersection-over-union."""
    xi1, yi1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    xi2, yi2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-9)

# sanity checks
print("identical boxes:   ", round(iou([0,0,10,10],[0,0,10,10]), 2), " (expect 1.0)")
print("half overlap:      ", round(iou([0,0,10,10],[5,0,15,10]), 3), " (expect 0.333)")
print("no overlap:        ", round(iou([0,0,10,10],[20,20,30,30]), 2), " (expect 0.0)")

In [ ]:
# visualize IoU on two boxes
def show_iou(b1, b2):
    fig, ax = plt.subplots(figsize=(4, 4))
    for b, c in [(b1, "blue"), (b2, "orange")]:
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                       fill=False, edgecolor=c, linewidth=2))
    ax.set_xlim(-2, 22); ax.set_ylim(-2, 22); ax.invert_yaxis()
    ax.set_title(f"IoU = {iou(b1, b2):.2f}"); ax.set_aspect("equal")
    plt.show()

show_iou([2, 2, 14, 14], [8, 6, 20, 18])

### Use IoU to score Part 1's predictions
Now we can measure our localization model properly: what fraction of its boxes have IoU ≥ 0.5 with the truth?

In [ ]:
def xywh_to_corners(b):
    return [b[0], b[1], b[0]+b[2], b[1]+b[3]]

_, pb_all = model.predict(X_test, verbose=0)
ious = [iou(xywh_to_corners(pb_all[i]), xywh_to_corners(yb_test[i])) for i in range(len(X_test))]
ious = np.array(ious)
print(f"mean IoU: {ious.mean():.3f}")
print(f"correct localizations (IoU >= 0.5): {(ious >= 0.5).mean():.1%}")

plt.hist(ious, bins=20, color="steelblue"); plt.axvline(0.5, color="red", ls="--", label="0.5 threshold")
plt.xlabel("IoU"); plt.ylabel("count"); plt.title("How well the model localizes"); plt.legend(); plt.show()

---
# Part 3 · Non-Max Suppression (NMS)

A real detector predicts **many overlapping boxes** for the same object. NMS keeps the best and removes the duplicates:

1. Discard boxes with low confidence.
2. Take the highest-confidence box, keep it.
3. Remove any remaining box that overlaps it too much (IoU ≥ threshold).
4. Repeat until none are left.

We implement exactly that.

In [ ]:
def non_max_suppression(boxes, scores, score_thresh=0.5, iou_thresh=0.5):
    """boxes: list of [x1,y1,x2,y2]; scores: confidences. Returns kept indices."""
    idxs = [i for i in range(len(boxes)) if scores[i] >= score_thresh]   # step 1
    idxs = sorted(idxs, key=lambda i: scores[i], reverse=True)
    keep = []
    while idxs:
        best = idxs.pop(0)          # step 2: highest score
        keep.append(best)
        # step 3: drop boxes overlapping the best one too much
        idxs = [i for i in idxs if iou(boxes[best], boxes[i]) < iou_thresh]
    return keep

In [ ]:
# demo: 5 boxes around 2 real objects (lots of duplicates), plus their scores
boxes = [[10,10,50,50],[12,14,52,54],[8,8,48,46],       # cluster A (same object)
         [70,60,110,100],[72,63,112,103]]                # cluster B (same object)
scores = [0.9, 0.75, 0.8, 0.95, 0.7]

kept = non_max_suppression(boxes, scores, score_thresh=0.5, iou_thresh=0.4)
print("kept boxes:", kept, "(5 noisy boxes -> the 2 real objects)")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, title, show in [(axes[0], "Before NMS (all boxes)", range(len(boxes))),
                        (axes[1], "After NMS (kept)", kept)]:
    ax.set_xlim(0, 130); ax.set_ylim(0, 120); ax.invert_yaxis(); ax.set_aspect("equal")
    for i in show:
        b = boxes[i]
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                     fill=False, edgecolor="red", linewidth=2))
        ax.text(b[0], b[1]-2, f"{scores[i]:.2f}", color="blue", fontsize=9)
    ax.set_title(title)
plt.tight_layout(); plt.show()

**NMS turned 5 noisy boxes into the 2 real detections.** Every object detector ends with this step. You've now built the three core pieces: **localization, IoU, and NMS.**

---
# Part 4 · YOLO — the real thing

Our Part-1 model handles **one** object. Real images have **many**. **YOLO** (You Only Look Once) divides the image into a grid, predicts boxes for every cell in a single pass, and uses NMS to clean up — all in real time.

Training YOLO needs huge labeled datasets, so we do what practitioners do: **use a pretrained YOLO** and run it. Watch it detect many objects at once — the payoff of everything above.

In [ ]:
# install Ultralytics YOLO (a quick download on Kaggle)
!pip install ultralytics --quiet

from ultralytics import YOLO
yolo = YOLO("yolov8n.pt")   # 'n' = nano: smallest & fastest, downloads on first use
# (newer families like yolo11n.pt / yolo26n.pt work too — same API)
print("YOLO loaded")

### Run YOLO on an image
- Point it at any photo (upload your own, or use a Kaggle dataset image).
- It returns boxes + class labels + confidence scores — already NMS-cleaned internally.

In [ ]:
# a sample image bundled with ultralytics (or replace with your own path)
IMAGE_PATH = "https://ultralytics.com/images/bus.jpg"

results = yolo(IMAGE_PATH, verbose=False)
r = results[0]

# YOLO can render the annotated image for us
annotated = r.plot()                      # BGR image with boxes drawn
plt.figure(figsize=(7, 9))
plt.imshow(annotated[..., ::-1])          # BGR -> RGB
plt.axis("off"); plt.title("YOLO detections")
plt.show()

In [ ]:
# read out what it found: label, confidence, box
names = r.names
print(f"found {len(r.boxes)} objects:\n")
for b in r.boxes:
    cls = names[int(b.cls[0])]
    conf = float(b.conf[0])
    x1, y1, x2, y2 = b.xyxy[0].tolist()
    print(f"  {cls:12s}  conf {conf:.2f}  box [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

### Connect it back
Everything YOLO just did, you now understand the pieces of:
- each detection has a **bounding box** — like your Part 1 box head,
- overlapping detections were removed with **NMS** — the function you wrote in Part 3,
- and "correct" detections are judged by **IoU** — which you implemented in Part 2.

YOLO isn't magic — it's these ideas, scaled up and made real-time.

## Your turn (solo task) ✍️

Pick at least two:
1. **Run YOLO on your own images** (upload a few) and list what it finds.
2. **Lower the confidence threshold**: `yolo(img, conf=0.1)` — do more (weaker) boxes appear?
3. **Change the NMS IoU threshold** in your Part-3 function (0.2 vs 0.7) on the demo boxes — how does the kept set change?
4. **Improve Part 1**: add a conv block or train longer, and check if the mean IoU goes up.
5. **Try a bigger YOLO** (`yolov8s.pt`) — more accurate, a little slower.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



## Detection vs the other vision tasks

Where object detection sits among the vision tasks you now know:

| Task | Output | Example |
|---|---|---|
| **Classification** | one label for the whole image | "this is a cat" |
| **Localization** | one label + one box | "a cat, here" (Part 1) |
| **Detection** | many labels + many boxes | "3 cars, 2 people, here, here..." (YOLO) |
| **Segmentation** | a label for every pixel | the exact outline of each object |

Detection is the workhorse of real-world vision: self-driving cars, surveillance, retail analytics, medical scans, robotics — anywhere a system must find *multiple* things and know *where* they are.

## Summary

- **Localization** = classification + a **box-regression head**, trained with a **combined loss** (cross-entropy + MSE). You built and trained it on shapes.
- **IoU** measures box overlap; "correct" usually means **IoU ≥ 0.5**. You implemented it and scored your model.
- **Non-max suppression** removes duplicate boxes, keeping the best per object. You implemented the exact algorithm.
- **YOLO** does detection for many objects in one real-time pass; you ran a pretrained model and read its outputs.
- The industrial detector is built from the **same three ideas** you coded by hand.

**Next:** a first look at **image segmentation** — labeling every pixel, the most detailed vision task of all.